In [ ]:
# PyTorch SegNet conversion
import os, time
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from skimage.io import imread
from skimage.transform import resize
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Overview
- SegNet for image segmentation
- Input: depth image (acquired via Kinect camera)
- Output: hand components

# Import libraries

In [ ]:
n_classes = 7
img_h = img_w = 96
def prepare_segnet_data(root_path):
    files = os.listdir(root_path); data_x = np.zeros((len(files), 1, img_h, img_w), dtype='float32'); data_y = np.zeros((len(files), img_h, img_w), dtype='int64')
    colors = [255, 43, 85, 128, 170, 212, 0]
    for i, filename in enumerate(files):
        image_path = os.path.join(root_path, filename); label_path = image_path.replace('depth', 'labels')
        data_x[i, 0] = resize(imread(image_path), (img_h, img_w), order=0, preserve_range=True)
        labels = resize(imread(label_path), (img_h, img_w), order=0, preserve_range=True)
        data_y[i] = np.argmin(np.abs(labels[..., None] - np.array(colors)), axis=-1)
    return torch.from_numpy(data_x), torch.from_numpy(data_y)

In [ ]:
class SegNet(nn.Module):
    def __init__(self, classes=7):
        super().__init__()
        self.encoder = nn.Sequential(nn.Conv2d(1, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(128, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU())
        self.decoder = nn.Sequential(nn.ConvTranspose2d(256, 128, 2, stride=2), nn.ReLU(), nn.ConvTranspose2d(128, 64, 2, stride=2), nn.ReLU(), nn.Conv2d(64, classes, 1))
    def forward(self, x): return self.decoder(self.encoder(x))
model = SegNet(n_classes).to(device)
print(model)

# (1) Data

#### Hand images: https://drive.google.com/file/d/1MRiBnDW9dE7mNb5bL-jzhmtTd2CSklvL/view?usp=sharing

In [ ]:
# Use CrossEntropyLoss with integer masks instead of one-hot categorical_crossentropy.
criterion = nn.CrossEntropyLoss(); optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
def train_segnet(x_train, y_train, x_test, y_test, epochs=15, batch_size=8):
    loader = DataLoader(TensorDataset(x_train.to(device), y_train.to(device)), batch_size=batch_size, shuffle=True)
    for epoch in range(epochs):
        model.train()
        for xb, yb in loader:
            optimizer.zero_grad(); loss = criterion(model(xb), yb); loss.backward(); optimizer.step()
        print(f'Epoch {epoch + 1}/{epochs}, loss={loss.item():.4f}')
    return model

In [ ]:
# Example after loading x_train/y_train/x_test/y_test: train_segnet(x_train, y_train, x_test, y_test)

# (2) Declare model

# (3) Train defined model
- In case your system runs out of memory (OOM), try to use smaller batch size

# (4) Evaluate trained model